In [ ]:
REPO_URL = "https://github.com/30nikamvaishnavi/autonomous-driving-perception-stack.git"
BRANCH = "main"
DATA_DIR = None  # Override to use a different runtime dataset path.
OUT_DIR = None  # Override to change where outputs and checkpoints are saved.
WANDB_PROJECT = "adpstack"
MODEL_PATH = None  # Optional .pt path for export/benchmark notebooks.


In [ ]:
import os
import subprocess
import sys
import time
from pathlib import Path

SMOKE_STARTED_AT = time.monotonic()
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = Path("/kaggle").exists()
if IS_COLAB == IS_KAGGLE:
    raise RuntimeError("Unsupported or ambiguous runtime: use Google Colab or Kaggle.")

if IS_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    runtime_root = Path("/content")
    data_default = Path("/content/drive/MyDrive/adpstack/data/bdd100k")
    output_default = Path("/content/drive/MyDrive/adpstack/outputs")
else:
    runtime_root = Path("/kaggle/working")
    data_default = Path("/kaggle/input")
    output_default = Path("/kaggle/working/outputs")

DATA_DIR = Path(DATA_DIR) if DATA_DIR is not None else data_default
OUT_DIR = Path(OUT_DIR) if OUT_DIR is not None else output_default
OUT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["WANDB_PROJECT"] = WANDB_PROJECT

REPO_DIR = runtime_root / "autonomous-driving-perception-stack"
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)
elif REPO_DIR.exists():
    raise FileExistsError(f"{REPO_DIR} exists but is not a Git checkout; remove/rename it and retry.")
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR / "src"))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[ml,export]"],
    cwd=REPO_DIR,
    check=True,
)
import torch

print(f"Runtime: {'Colab' if IS_COLAB else 'Kaggle'}")
print(f"Repository: {REPO_DIR}")
print(f"Data: {DATA_DIR}")
print(f"Outputs: {OUT_DIR}")
print(f"PyTorch: {torch.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU is attached. Select a T4 GPU runtime in Colab or enable a GPU accelerator in Kaggle, then restart."
    )
print(f"GPU: {torch.cuda.get_device_name(0)}")


# 03 - ONNX export + TensorRT benchmark on a Colab T4

In [ ]:
import shutil
from pathlib import Path

from ultralytics import YOLO

if MODEL_PATH is None:
    model_candidates = [OUT_DIR / "runs/detect/yolo11n/weights/best.pt"]
    if IS_KAGGLE:
        model_candidates.extend(sorted(DATA_DIR.glob("**/best.pt")))
    MODEL_PATH = next((path for path in model_candidates if path.is_file()), None)
if MODEL_PATH is None:
    raise FileNotFoundError("Set MODEL_PATH to a trained best.pt or attach it as Kaggle input.")
MODEL_PATH = Path(MODEL_PATH)
WORKING_MODEL = OUT_DIR / "models" / "yolo11n_best.pt"
WORKING_MODEL.parent.mkdir(parents=True, exist_ok=True)
if MODEL_PATH.resolve() != WORKING_MODEL.resolve():
    shutil.copy2(MODEL_PATH, WORKING_MODEL)
ONNX_PATH = Path(YOLO(str(WORKING_MODEL)).export(format="onnx", imgsz=640, opset=17, simplify=True))
print(f"ONNX model: {ONNX_PATH}")

In [ ]:
import subprocess

from pathlib import Path


def find_bdd_root(base: Path) -> Path:
    candidates = [base]
    if base.is_dir():
        candidates.extend(path for path in sorted(base.iterdir()) if path.is_dir())
    for candidate in candidates:
        if (candidate / "images/val").is_dir():
            return candidate
    raise FileNotFoundError(f"No BDD100K validation images found under {base}.")


BDD_ROOT = find_bdd_root(DATA_DIR)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "tensorrt", "onnxruntime-gpu"],
    check=True,
)
ENGINE_DIR = OUT_DIR / "engines"
ENGINE_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "adpstack.export.to_tensorrt", "--onnx", str(ONNX_PATH),
     "--precision", "fp16", "--out", str(ENGINE_DIR / "yolo_fp16.engine")],
    cwd=REPO_DIR,
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "adpstack.export.to_tensorrt", "--onnx", str(ONNX_PATH),
     "--precision", "int8", "--calib-dir", str(BDD_ROOT / "images/val"),
     "--out", str(ENGINE_DIR / "yolo_int8.engine")],
    cwd=REPO_DIR,
    check=True,
)

In [ ]:
subprocess.run(
    [sys.executable, "-m", "adpstack.export.benchmark",
     "--torch-weights", str(WORKING_MODEL), "--onnx", str(ONNX_PATH),
     "--engines", f"trt_fp16={ENGINE_DIR / 'yolo_fp16.engine'}",
     f"trt_int8={ENGINE_DIR / 'yolo_int8.engine'}", "--iters", "500",
     "--out", str(OUT_DIR / "benchmark")],
    cwd=REPO_DIR,
    check=True,
)

## INT8 accuracy drop
Run `yolo val model=outputs/yolo_int8.engine data=... ` (Ultralytics supports .engine) and record mAP50-95 vs FP32 in docs/results.md.